# Phase 7 - Step 2: all experiments in one notebook (re-ranking BM25's top 100 with Phase-1 models)

**Pipeline (same for every experiment)**

1. **Stage 1 - BM25** (best P7_01 setting, dev Micro-F1@7 0.4247): top 100 candidates per query
   (73% of all relevant cases are inside the dev top 100 = the ceiling of any re-ranker on top of it).
2. **Stage 2 - cross-encoder**: a Phase-1 model reads `[candidate: first ~700 tokens] EOS [query: words around its
<CITATION> markers, <= 320 tokens] EOS` and outputs one relevance score (attention pooling over the query part).
   Trained with a **listwise contrastive loss**: group = 1 relevant case + 6 BM25 hard negatives + 1 random case,
   softmax cross-entropy with the relevant one as the correct answer.
3. **Fusion**: `s = w * z(model) + (1 - w) * z(BM25)` per query, candidates newer than the query go to the
   bottom (year filter), top 7 submitted. `w` tuned on dev; `w = 1` = model alone (also reported).

| #   | Phase-1 model                       | technique (Phase-7 list)                                 | why                                                 |
| --- | ----------------------------------- | -------------------------------------------------------- | --------------------------------------------------- |
| 1   | A2 (decoder + copy)                 | **LoRMA** r8 (compulsory)                                | multiplicative low-rank update, 0.6% of the weights |
| 2   | A2                                  | **full fine-tuning**                                     | strongest model = KD teacher                        |
| 3   | D encoder (4 layers, bidirectional) | full FT + **KD** from #2                                 | smaller, faster student                             |
| 3c  | D encoder                           | same, **no teacher** (control, dev only - do not upload) | shows what KD adds                                  |
| 4   | A (plain decoder)                   | **IA3**                                                  | only scaling vectors (~0.03% of the weights)        |
| 5   | A2                                  | **partial (layer-wise) FT**, top 4 of 10 blocks          | cheaper than full FT                                |
| 6   | #1-#5                               | **score ensemble** (+ BM25)                              | combines the models                                 |

Everything is cached: a finished run is only loaded (training checkpoints every 250 steps + resume, scores,
report, submission). Each experiment cell catches its own errors, so one failure does not stop the overnight run.


In [1]:
import os, sys, json, time, traceback
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / ".gitignore").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import torch
import pcr_utils as pu
import pcr_models as pm
import pcr_experiment as pe
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)

# QUICK = True only for the CPU test with tiny fake models (few queries, short inputs). Keep False on the GPU.
QUICK = os.environ.get("P7_QUICK") == "1"
Q = dict(train_limit=12, depth=12, max_len=256, q_max=96, epochs=2, ckpt_every=2, groups_per_step=2, infer_rows=8) if QUICK else {}
print("project root:", ROOT, "| QUICK test mode" if QUICK else "")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")

RESULTS = {}
def run(key, cfg):
    # one experiment; errors are printed and the notebook continues with the next cell
    try:
        e = pe.Experiment(ROOT, {**cfg, **Q})
        RESULTS[key] = e.run()
        return e
    except Exception:
        traceback.print_exc()
        print(f"\n!!! experiment {key} FAILED - continuing with the next one")
        return None

c:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


project root: C:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project 
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


## 0. Stage 1 (BM25 top 200 for every query) and the token cache - once for all experiments


In [2]:
P = pe.prepare(ROOT)
q0 = next(iter(P["top"]["dev"]))
print("example pair layout (query", q0, "+ its BM25 top-1 candidate):")
ids, qs = pm.make_pair(P["cand"][P["top"]["dev"][q0]["ids"][0]], P["query"][q0], 1024, 320)
print(f"  {len(ids)} tokens: candidate {qs - 1}, EOS, query {len(ids) - qs - 1}, EOS")

BM25 first stage, setting: {'c_words': 2048, 'k1': 8.0, 'b': 1.0, 'mode': 'cit', 'year_margin': 0}


BM25 first stage on dev: Micro-F1@7 0.4247  (P7_01 best: 0.4247)
share of gold pairs inside the BM25 top 100 (= best possible after re-ranking): {'train': 0.7097, 'dev': 0.7307}


tokenise query snippets: 100%|██████████| 5/5 [00:00<00:00, 24.91it/s]

tokens: 5,825 candidates (start, median 1500), 1,182 query snippets (median 410, p90 512)
example pair layout (query 0000012637 + its BM25 top-1 candidate):
  1024 tokens: candidate 702, EOS, query 320, EOS


## Experiment 1 - A2 + LoRMA r8 (compulsory)

`h = (I + (alpha/r) B A) W0 x`: the frozen Phase-1 output `W0 x` is multiplied by a learned low-rank matrix.


In [3]:
E1 = run("1_A2_lorma", dict(phase1="A2", method="lorma", rank=8, alpha=16))

experiment: A2_lormar8_causal_L1024 | LoRMA r8 | device cuda | precision torch.bfloat16
memory check: {'params_M': 71.12, 'trainable_M': 1.066, 'trainable_%': 1.5, 'group': '8 pairs x 1024 tokens', 'sec_per_group (first, incl. warm-up)': 1.01, 'groups_per_epoch': 3791, 'peak_gb': 3.56, 'phase1_load': {'arch': 'A2', 'layers': 10, 'loaded_tensors': 72, 'dropped_tensors': 5, 'dropped_groups': ['copy_gate', 'copy_k', 'copy_q', 'lm_head']}}


epoch 1/2: 100%|██████████| 948/948 [20:44<00:00,  1.31s/it, loss=1.6332]


epoch 1 | train_loss 1.8085 | train_min 20.8 | dev_f1 0.4314 | dev_f1_neural_only 0.289 | dev_w 0.35 | eval_min 3.2  <- best


epoch 2/2: 100%|██████████| 948/948 [20:43<00:00,  1.31s/it, loss=1.3196]


epoch 2 | train_loss 1.4794 | train_min 20.7 | dev_f1 0.4314 | dev_f1_neural_only 0.2957 | dev_w 0.4 | eval_min 3.3
early stop: no improvement for 1 epoch(s)
training finished -> loaded best epoch 1
dev Micro-F1@7: BM25 0.4247 | neural only 0.289 | fused 0.4314 (w 0.35) | queries better / worse than BM25: 14 / 9
[cache] report: dev fused 0.4314 (w 0.35), neural only 0.289


test: 23,700 pairs scored in 389 s (61,714 tokens/s)
submission checked: {'queries': 237, 'ids_per_query': 7, 'distinct_ids_used': 996} -> C:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\checkpoints\phase7\A2_lormar8_causal_L1024\submission.zip


## Experiment 2 - A2 full fine-tuning (KD teacher)


In [4]:
E2 = run("2_A2_full", dict(phase1="A2", method="full", lr=3e-5))

experiment: A2_full_causal_L1024 | full fine-tuning | device cuda | precision torch.bfloat16
memory check: {'params_M': 70.06, 'trainable_M': 70.056, 'trainable_%': 100.0, 'group': '8 pairs x 1024 tokens', 'sec_per_group (first, incl. warm-up)': 0.34, 'groups_per_epoch': 3791, 'peak_gb': 3.49, 'phase1_load': {'arch': 'A2', 'layers': 10, 'loaded_tensors': 72, 'dropped_tensors': 5, 'dropped_groups': ['copy_gate', 'copy_k', 'copy_q', 'lm_head']}}


epoch 1/2: 100%|██████████| 948/948 [18:50<00:00,  1.19s/it, loss=1.5697]


epoch 1 | train_loss 1.7442 | train_min 18.8 | dev_f1 0.4336 | dev_f1_neural_only 0.2835 | dev_w 0.2 | eval_min 2.2  <- best


epoch 2/2: 100%|██████████| 948/948 [18:47<00:00,  1.19s/it, loss=0.9333]


epoch 2 | train_loss 1.1071 | train_min 18.8 | dev_f1 0.4291 | dev_f1_neural_only 0.2813 | dev_w 0.3 | eval_min 2.2
early stop: no improvement for 1 epoch(s)
training finished -> loaded best epoch 1
dev Micro-F1@7: BM25 0.4247 | neural only 0.2835 | fused 0.4336 (w 0.2) | queries better / worse than BM25: 12 / 5
[cache] report: dev fused 0.4336 (w 0.2), neural only 0.2835


test: 23,700 pairs scored in 258 s (92,957 tokens/s)
submission checked: {'queries': 237, 'ids_per_query': 7, 'distinct_ids_used': 996} -> C:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\checkpoints\phase7\A2_full_causal_L1024\submission.zip


## Experiment 3 - D encoder (4 layers) + knowledge distillation from #2

The student sees the same group as the teacher and matches the teacher's softmax over the group (T = 2) as well
as the gold answer: `loss = 0.5 * CE(gold) + 0.5 * T^2 * KL(teacher || student)`.


In [5]:
TEACHER = pe.run_name({**pe.DEFAULTS, "phase1": "A2", "method": "full", "causal": True, **Q})
print("teacher:", TEACHER)
E3 = run("3_D_kd", dict(phase1="D", method="full", lr=5e-5, epochs=3 if not QUICK else 2, kd_teacher=TEACHER, kd_alpha=0.5, kd_T=2.0))

teacher: A2_full_causal_L1024
experiment: D_full_bidir_L1024_kd2 | full fine-tuning + KD from A2_full_causal_L1024 | device cuda | precision torch.bfloat16
teacher A2_full_causal_L1024 loaded (dev F1 fused 0.4336)
memory check: {'params_M': 40.31, 'trainable_M': 40.312, 'trainable_%': 100.0, 'group': '8 pairs x 1024 tokens', 'sec_per_group (first, incl. warm-up)': 0.24, 'groups_per_epoch': 3791, 'peak_gb': 1.9, 'phase1_load': {'arch': 'D_encdec', 'layers': 4, 'loaded_tensors': 30, 'dropped_tensors': 61, 'dropped_groups': ['copy_gate', 'copy_k', 'copy_q', 'dec', 'dec_norm', 'lm_head']}}
teacher A2_full_causal_L1024 loaded (dev F1 fused 0.4336)


epoch 1/3: 100%|██████████| 948/948 [13:48<00:00,  1.14it/s, loss=1.7519]


epoch 1 | train_loss 1.6605 | train_min 13.8 | dev_f1 0.4358 | dev_f1_neural_only 0.0956 | dev_w 0.25 | eval_min 0.9  <- best


epoch 2/3: 100%|██████████| 948/948 [13:47<00:00,  1.15it/s, loss=1.5117]


epoch 2 | train_loss 1.5127 | train_min 13.8 | dev_f1 0.438 | dev_f1_neural_only 0.1078 | dev_w 0.3 | eval_min 0.9  <- best


epoch 3/3: 100%|██████████| 948/948 [13:47<00:00,  1.15it/s, loss=1.3284]


epoch 3 | train_loss 1.3648 | train_min 13.8 | dev_f1 0.4325 | dev_f1_neural_only 0.1123 | dev_w 0.3 | eval_min 0.9
early stop: no improvement for 1 epoch(s)
training finished -> loaded best epoch 2
dev Micro-F1@7: BM25 0.4247 | neural only 0.1078 | fused 0.438 (w 0.3) | queries better / worse than BM25: 17 / 6
[cache] report: dev fused 0.438 (w 0.3), neural only 0.1078


test: 23,700 pairs scored in 113 s (212,962 tokens/s)
submission checked: {'queries': 237, 'ids_per_query': 7, 'distinct_ids_used': 979} -> C:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\checkpoints\phase7\D_full_bidir_L1024_kd2\submission.zip


## Experiment 3c - D encoder without teacher (control for KD; dev only, do not upload)


In [6]:
E3c = run("3c_D_control", dict(phase1="D", method="full", lr=5e-5, epochs=3 if not QUICK else 2))

experiment: D_full_bidir_L1024 | full fine-tuning | device cuda | precision torch.bfloat16
memory check: {'params_M': 40.31, 'trainable_M': 40.312, 'trainable_%': 100.0, 'group': '8 pairs x 1024 tokens', 'sec_per_group (first, incl. warm-up)': 0.14, 'groups_per_epoch': 3791, 'peak_gb': 1.53, 'phase1_load': {'arch': 'D_encdec', 'layers': 4, 'loaded_tensors': 30, 'dropped_tensors': 61, 'dropped_groups': ['copy_gate', 'copy_k', 'copy_q', 'dec', 'dec_norm', 'lm_head']}}


epoch 1/3: 100%|██████████| 948/948 [08:33<00:00,  1.85it/s, loss=1.9888]


epoch 1 | train_loss 2.0366 | train_min 8.6 | dev_f1 0.4369 | dev_f1_neural_only 0.0945 | dev_w 0.25 | eval_min 0.9  <- best


epoch 2/3: 100%|██████████| 948/948 [08:32<00:00,  1.85it/s, loss=1.9120]


epoch 2 | train_loss 1.9343 | train_min 8.5 | dev_f1 0.4325 | dev_f1_neural_only 0.1012 | dev_w 0.25 | eval_min 0.9
early stop: no improvement for 1 epoch(s)
training finished -> loaded best epoch 1
dev Micro-F1@7: BM25 0.4247 | neural only 0.0945 | fused 0.4369 (w 0.25) | queries better / worse than BM25: 14 / 4
[cache] report: dev fused 0.4369 (w 0.25), neural only 0.0945


test: 23,700 pairs scored in 113 s (212,746 tokens/s)
submission checked: {'queries': 237, 'ids_per_query': 7, 'distinct_ids_used': 998} -> C:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\checkpoints\phase7\D_full_bidir_L1024\submission.zip


## Experiment 4 - A + IA3

Only vectors that rescale the keys, values and feed-forward activations (start at 1 = Phase-1 model).


In [7]:
E4 = run("4_A_ia3", dict(phase1="A", method="ia3"))

experiment: A_ia3_causal_L1024 | IA3 | device cuda | precision torch.bfloat16
memory check: {'params_M': 70.09, 'trainable_M': 0.031, 'trainable_%': 0.04, 'group': '8 pairs x 1024 tokens', 'sec_per_group (first, incl. warm-up)': 0.37, 'groups_per_epoch': 3791, 'peak_gb': 2.9, 'phase1_load': {'arch': 'A_decoder', 'layers': 10, 'loaded_tensors': 72, 'dropped_tensors': 1, 'dropped_groups': ['lm_head']}}


epoch 1/2: 100%|██████████| 948/948 [22:35<00:00,  1.43s/it, loss=1.9371]


epoch 1 | train_loss 2.0174 | train_min 22.6 | dev_f1 0.4314 | dev_f1_neural_only 0.1467 | dev_w 0.25 | eval_min 3.0  <- best


epoch 2/2: 100%|██████████| 948/948 [22:35<00:00,  1.43s/it, loss=1.8084]


epoch 2 | train_loss 1.8792 | train_min 22.6 | dev_f1 0.4314 | dev_f1_neural_only 0.1845 | dev_w 0.15 | eval_min 3.0
early stop: no improvement for 1 epoch(s)
training finished -> loaded best epoch 1
dev Micro-F1@7: BM25 0.4247 | neural only 0.1467 | fused 0.4314 (w 0.25) | queries better / worse than BM25: 11 / 5
[cache] report: dev fused 0.4314 (w 0.25), neural only 0.1467


test: 23,700 pairs scored in 362 s (66,327 tokens/s)
submission checked: {'queries': 237, 'ids_per_query': 7, 'distinct_ids_used': 991} -> C:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\checkpoints\phase7\A_ia3_causal_L1024\submission.zip


## Experiment 5 - A2 partial (layer-wise) fine-tuning: top 4 of 10 blocks


In [8]:
E5 = run("5_A2_partial4", dict(phase1="A2", method="partial", top_blocks=4))

experiment: A2_partial4_causal_L1024 | partial (layer-wise) FT, top 4 blocks | device cuda | precision torch.bfloat16
memory check: {'params_M': 70.06, 'trainable_M': 19.832, 'trainable_%': 28.31, 'group': '8 pairs x 1024 tokens', 'sec_per_group (first, incl. warm-up)': 0.17, 'groups_per_epoch': 3791, 'peak_gb': 1.63, 'phase1_load': {'arch': 'A2', 'layers': 10, 'loaded_tensors': 72, 'dropped_tensors': 5, 'dropped_groups': ['copy_gate', 'copy_k', 'copy_q', 'lm_head']}}


epoch 1/2: 100%|██████████| 948/948 [10:33<00:00,  1.50it/s, loss=1.5190]


epoch 1 | train_loss 1.7313 | train_min 10.6 | dev_f1 0.4347 | dev_f1_neural_only 0.2868 | dev_w 0.4 | eval_min 2.1  <- best


epoch 2/2: 100%|██████████| 948/948 [10:33<00:00,  1.50it/s, loss=1.1030]


epoch 2 | train_loss 1.2639 | train_min 10.6 | dev_f1 0.4325 | dev_f1_neural_only 0.3024 | dev_w 0.25 | eval_min 2.1
early stop: no improvement for 1 epoch(s)
training finished -> loaded best epoch 1
dev Micro-F1@7: BM25 0.4247 | neural only 0.2868 | fused 0.4347 (w 0.4) | queries better / worse than BM25: 16 / 9
[cache] report: dev fused 0.4347 (w 0.4), neural only 0.2868


test: 23,700 pairs scored in 258 s (92,972 tokens/s)
submission checked: {'queries': 237, 'ids_per_query': 7, 'distinct_ids_used': 1009} -> C:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\checkpoints\phase7\A2_partial4_causal_L1024\submission.zip


## Experiment 6 - score ensemble of #1-#5 (+ BM25)

Each model's scores are z-normalised per query and averaged, then fused with BM25 (weight tuned on dev).


In [9]:
members = [e.name for e in (E1, E2, E3, E4, E5) if e is not None and (e.dir / "scores_test.npz").exists()]
print("members:", members)
try:
    R6 = pe.ensemble(ROOT, members, out_name="ensemble_1to5" + ("_quick" if QUICK else ""), depth=Q.get("depth", 100)) if len(members) >= 2 else None
except Exception:
    traceback.print_exc()

members: ['A2_lormar8_causal_L1024', 'A2_full_causal_L1024', 'D_full_bidir_L1024_kd2', 'A_ia3_causal_L1024', 'A2_partial4_causal_L1024']
ensemble dev Micro-F1@7: neural only 0.2824 | fused 0.4369 (w 0.35) | members {'A2_lormar8_causal_L1024': 0.4314, 'A2_full_causal_L1024': 0.4336, 'D_full_bidir_L1024_kd2': 0.438, 'A_ia3_causal_L1024': 0.4314, 'A2_partial4_causal_L1024': 0.4347}
submission checked: {'queries': 237, 'ids_per_query': 7, 'distinct_ids_used': 1010} -> C:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\checkpoints\phase7\ensemble_1to5\submission.zip


## Comparison (dev Micro-F1@7; BM25 alone = 0.4247)

`dev_neural_only_f1` = the re-ranker alone (w = 1), `dev_fused_f1` = with BM25 at the dev-tuned weight `w`.


In [10]:
display(pe.compare(ROOT))
for key, r in RESULTS.items():
    if r:
        print(f"\n{key}: by number of gold cases (model vs BM25):", r["by_n_gold"])
        print(f"   re-rank depth: {r['by_depth']}")
        print(f"   queries better / worse than BM25: {r['queries_better_than_bm25']} / {r['queries_worse_than_bm25']}")
        print(f"   epochs: {r['history']}")

,technique,trainable_M,trainable_%,best_epoch,dev_neural_only_f1,w,dev_fused_f1,dev_fused_P,dev_fused_R,min_per_epoch,infer_tokens_per_s,peak_gb_train
name,,,,,,,,,,,,
A2_lormar8_causal_L1024,LoRMA r8,1.066,1.50,1.0,0.2890,0.35,0.4314,0.4697,0.3988,"[20.8, 20.7]",61598.0,3.56
A2_full_causal_L1024,full fine-tuning,70.056,100.00,1.0,0.2835,0.20,0.4336,0.4722,0.4008,"[18.8, 18.8]",92892.0,3.49
D_full_bidir_L1024_kd2,full fine-tuning + KD from A2_full_causal_L1024,40.312,100.00,2.0,0.1078,0.30,0.4380,0.4770,0.4049,"[13.8, 13.8, 13.8]",212964.0,1.90
D_full_bidir_L1024,full fine-tuning,40.312,100.00,1.0,0.0945,0.25,0.4369,0.4758,0.4039,"[8.6, 8.5]",212785.0,1.53
A_ia3_causal_L1024,IA3,0.031,0.04,1.0,0.1467,0.25,0.4314,0.4697,0.3988,"[22.6, 22.6]",66273.0,2.90
A2_partial4_causal_L1024,"partial (layer-wise) FT, top 4 blocks",19.832,28.31,1.0,0.2868,0.40,0.4347,0.4734,0.4018,"[10.6, 10.6]",93025.0,1.63
ensemble_1to5,"score ensemble of A2_lormar8_causal_L1024, A2_...",NaN,NaN,NaN,0.2824,0.35,0.4369,0.4758,0.4039,None,NaN,NaN



1_A2_lorma: by number of gold cases (model vs BM25): {'1': {'queries': 15, 'model': 0.1833, 'bm25': 0.1833}, '2-3': {'queries': 25, 'model': 0.3534, 'bm25': 0.3362}, '4-7': {'queries': 39, 'model': 0.5331, 'bm25': 0.5207}, '8+': {'queries': 39, 'model': 0.4299, 'bm25': 0.4278}}
   re-rank depth: {20: {'w': 0.5, 'micro_f1': 0.4302, 'micro_p': 0.4685, 'micro_r': 0.3977}, 50: {'w': 0.5, 'micro_f1': 0.4314, 'micro_p': 0.4697, 'micro_r': 0.3988}, 100: {'w': 0.35, 'micro_f1': 0.4314, 'micro_p': 0.4697, 'micro_r': 0.3988}}
   queries better / worse than BM25: 14 / 9
   epochs: [{'epoch': 1, 'train_loss': 1.8085, 'train_min': 20.8, 'dev_f1': 0.4314, 'dev_f1_neural_only': 0.289, 'dev_w': 0.35, 'eval_min': 3.2}, {'epoch': 2, 'train_loss': 1.4794, 'train_min': 20.7, 'dev_f1': 0.4314, 'dev_f1_neural_only': 0.2957, 'dev_w': 0.4, 'eval_min': 3.3}]

2_A2_full: by number of gold cases (model vs BM25): {'1': {'queries': 15, 'model': 0.1833, 'bm25': 0.1833}, '2-3': {'queries': 25, 'model': 0.3534, 'bm2

## KD: size / speed / quality (student #3 vs teacher #2 vs control #3c)


In [11]:
rows = []
for key in ("2_A2_full", "3_D_kd", "3c_D_control"):
    r = RESULTS.get(key)
    if r:
        rows.append({"run": key, "params trainable (M)": r["trainable_M"], "dev fused F1": r["dev_fused_f1"],
                     "dev model-only F1": r["dev_neural_only_f1"], "tokens/s (inference)": r["infer_tokens_per_s"],
                     "min/epoch": r["min_per_epoch"], "peak GB (train)": r["peak_gb_train"]})
display(pd.DataFrame(rows).set_index("run") if rows else "no KD results yet")

,params trainable (M),dev fused F1,dev model-only F1,tokens/s (inference),min/epoch,peak GB (train)
run,,,,,,
2_A2_full,70.056,0.4336,0.2835,92892,"[18.8, 18.8]",3.49
3_D_kd,40.312,0.4380,0.1078,212964,"[13.8, 13.8, 13.8]",1.90
3c_D_control,40.312,0.4369,0.0945,212785,"[8.6, 8.5]",1.53
